# Exercise 2: Apply Pattern for Missing Values Handling

## Objective

- Learn the Strategy Pattern: one interface, many interchangeable behaviors.
- Practice inheritance and polymorphism.

## Concept

Different strategies can be used to handle missing data, such as dropping missing values, filling them with the mean, or filling them with the mode.

The Strategy Pattern allows us to switch between these methods without changing the main code.

## Instructions:
1. Create an abstract class MissingValueStrategy with a method handle(df).
2. Create subclasses:
- oDropMissing
- oFillMean
- oFillMode
3. Create a DataCleaner class that applies the chosen strategy.

In [7]:
import pandas as pd

df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")
print(df.head())

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


In [1]:
# 1. Create an abstract class MissingValueStrategy with a method handle(df).
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

In [2]:
# 2. Create subclasses:
# DropMissing
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()

In [8]:
df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")

cleaner = DataCleaner(DropMissing())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0


In [5]:
# FillMean
class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        numeric_cols = result.select_dtypes(include="number").columns
        result[numeric_cols] = result[numeric_cols].fillna(
            result[numeric_cols].mean()
        )
        return result

In [9]:
df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")

cleaner = DataCleaner(FillMean())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0


In [4]:
# FillMode
class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            if result[column].isna().any():
                mode_values = result[column].mode()

                if not mode_values.empty:
                    result[column] = result[column].fillna(
                        mode_values.iloc[0]
                    )

        return result

In [10]:
df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")

cleaner = DataCleaner(FillMode())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


In [11]:
# 3. Create a DataCleaner class that applies the chosen strategy.
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)